# Generate Report

In [0]:
import json
from datetime import datetime
from pyspark.sql import functions as F

# ============================================================
# Gather row counts across every layer
# ============================================================
tables_to_check = {
    "bronze": ["customers", "products", "stores", "sales_reps", "sales_orders", "order_items", "returns"],
    "silver": ["customers", "products", "stores", "sales_reps", "sales_orders", "order_items", "returns", "order_items_quarantine"],
    "gold": ["dim_date", "dim_category", "dim_customer", "dim_product", "dim_store", "dim_sales_rep", "fact_sales", "fact_returns"]
}

report = {"run_timestamp": str(datetime.now()), "layers": {}}

for layer, tables in tables_to_check.items():
    report["layers"][layer] = {}
    for t in tables:
        try:
            count = spark.table(f"azuresalesdatabricks.{layer}.{t}").count()
            report["layers"][layer][t] = count
        except Exception as e:
            report["layers"][layer][t] = f"ERROR: {str(e)}"

# ============================================================
# Data quality checks
# ============================================================
report["quality_checks"] = {
    "fact_sales_broken_keys": spark.sql("""
        SELECT COUNT(*) AS c FROM azuresalesdatabricks.gold.fact_sales
        WHERE customer_key IS NULL OR product_key IS NULL OR store_key IS NULL
           OR rep_key IS NULL OR date_key IS NULL
    """).collect()[0]["c"],
    "order_items_quarantined": spark.table("azuresalesdatabricks.silver.order_items_quarantine").count()
}

# ============================================================
# Print a readable summary in the notebook output
# ============================================================
print("=" * 50)
print(f"PIPELINE RUN REPORT — {report['run_timestamp']}")
print("=" * 50)
for layer, tables in report["layers"].items():
    print(f"\n{layer.upper()}:")
    for t, c in tables.items():
        print(f"  {t}: {c}")
print(f"\nQUALITY CHECKS:")
print(f"  fact_sales broken keys: {report['quality_checks']['fact_sales_broken_keys']}")
print(f"  order_items quarantined: {report['quality_checks']['order_items_quarantined']}")
print("=" * 50)

# Pass the report out for ADF/downstream consumption
dbutils.notebook.exit(json.dumps(report))